# Experimento con el LLM: ¿respeta las reglas de producción?

**Responsable:** Persona 5

Se le dan al LLM las reglas de nuestra gramática (`spanish/grammar.py`, sección 1.2 de
`docs/gramatica_es.md`), el léxico y las formas de árbol, y se le pide responder **solo** con el JSON de
la sección 2.6. El código del experimento está en `experiments/` y este notebook solo lo usa:

| Archivo | Qué hace |
| --- | --- |
| `experiments/prompts.py` | arma los dos prompts y lee la respuesta |
| `experiments/llm_client.py` | llama al proveedor (intercambiable) y guarda el log |
| `experiments/llm_experiment.py` | corre todas las oraciones y guarda `results/llm_*.jsonl` |
| `experiments/llm_analysis.py` | métricas propias del LLM (reglas frente a español, árboles, formato, costo) |

| Versión | Prompt | Salida |
| --- | --- | --- |
| `llm_zero` | solo reglas, léxico y formas de árbol | `results/llm_zero.jsonl` |
| `llm_few` | lo mismo más los ejemplos resueltos de `prompts.EXAMPLES` | `results/llm_few.jsonl` |

Todos los prompts y respuestas crudas quedan en `results/logs/llm.jsonl` (entregable obligatorio). Si un
`id` y `sistema` ya están en el log, se reutiliza la respuesta **sin volver a llamar a la API**: correr el
notebook otra vez no cuesta nada, y una corrida interrumpida sigue donde quedó.

**Antes de correrlo** (una sola vez):

1. `pip install google-genai` en el `.venv` del proyecto (o el SDK del proveedor elegido).
2. Crear `.env` en la raíz del repositorio (ya está en `.gitignore`) con la línea
   `GOOGLE_API_KEY=...`. **Nunca** poner la clave en el notebook.

Preguntas a responder (van al informe):

1. ¿El LLM respeta las reglas o se desvía hacia su conocimiento del español?
2. ¿Dónde acierta y dónde falla?
3. ¿El few-shot mejora la precisión?

In [ ]:
# Configuración: el notebook vive en notebooks/, el código del proyecto en la raíz.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from evaluation.formats import load_predictions, load_test_set, results_path

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 100)
FIGURES = ROOT / "results" / "figures"
REPORTS = ROOT / "results" / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)
items = load_test_set()
by_id = {i.id: i for i in items}
print(f"{len(items)} oraciones en el conjunto de prueba")

## 0. Configuración del experimento

Anotar en el informe el **modelo exacto** y la **fecha** (están en cada línea del log).

- `PROVIDER`: `"gemini"`, `"anthropic"` o `"manual"` (muestra el prompt y se pega la respuesta de cualquier
  chat, sin clave).
- `TEMPERATURE = 0` para que el experimento se pueda repetir.
- `PAUSE`: segundos antes de cada llamada **nueva**. La capa gratuita de Gemini limita las llamadas por
  minuto; si aun así hay un error 429, el cliente espera y reintenta solo.
- `PILOT`: con un número, hace solo esa cantidad de llamadas nuevas por versión y **no** guarda
  `results/llm_*.jsonl`. Sirve para revisar las primeras respuestas antes de gastar en las 146. Las
  respuestas quedan en el log y la corrida completa las reutiliza. Con `None`, corre todo.

In [ ]:
PROVIDER = "gemini"
MODEL = "gemini-3.5-flash-lite"   # la capa gratuita de gemini-3.8-flash solo da 20 llamadas por día
TEMPERATURE = 0
PAUSE = 7          # s; la capa gratuita limita las llamadas por minuto
PILOT = 3          # None = las 73 oraciones

from experiments.llm_client import api_key, read_log

KEY_NAME = {"gemini": "GOOGLE_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}.get(PROVIDER)
if KEY_NAME:
    try:
        api_key(KEY_NAME)          # solo comprueba que exista; no se muestra
        print(f"{KEY_NAME}: encontrada")
    except KeyError:
        print(f"Falta {KEY_NAME} en .env: solo se pueden reutilizar las respuestas que ya estén en el log.")
log = read_log()
print(f"Log: {len(log)} llamadas guardadas", {s: sum(e['sistema'] == s for e in log) for s in ('llm_zero', 'llm_few')})

### Los dos prompts

El prompt es el mismo para todas las oraciones; solo cambia la oración del final. El few-shot agrega los
ejemplos resueltos, que no son oraciones del conjunto de prueba (lo comprueba la celda).

In [ ]:
from experiments import prompts

zero_prompt, few_prompt = prompts.zero_shot(items[0].sentence), prompts.few_shot(items[0].sentence)
test_sentences = {i.sentence for i in items}
example_sentences = [sentence for sentence, _ in prompts.EXAMPLES]
assert not set(example_sentences) & test_sentences, "un ejemplo del few-shot está en el conjunto de prueba"
print(f"Zero-shot: {len(zero_prompt):,} caracteres | few-shot: {len(few_prompt):,} caracteres "
      f"({len(example_sentences)} ejemplos: {'; '.join(example_sentences)})")
print("=" * 100)
print(zero_prompt)

## 1. Zero-shot (solo reglas)

In [ ]:
from experiments import llm_experiment

llm_zero = llm_experiment.run("llm_zero", items, provider=PROVIDER, model=MODEL,
                              temperature=TEMPERATURE, pause=PAUSE, limit=PILOT)

## 2. Few-shot (reglas y ejemplos)

In [ ]:
llm_few = llm_experiment.run("llm_few", items, provider=PROVIDER, model=MODEL,
                             temperature=TEMPERATURE, pause=PAUSE, limit=PILOT)

In [ ]:
# Revisión rápida de las respuestas (sirve también para la prueba piloto).
from experiments import llm_analysis

pilot = pd.DataFrame([{**row, "sistema": s}
                      for s, preds in (("llm_zero", llm_zero), ("llm_few", llm_few))
                      for row in llm_analysis.per_sentence(items, preds)])
pilot[["sistema", "id", "oracion", "esperado", "llm", "acierta", "arbol_en_gramatica"]]

## 3. Análisis

A partir de aquí se usan los archivos guardados (`results/llm_*.jsonl`), así que hace falta la corrida
completa (`PILOT = None`). Las métricas comunes a todos los sistemas salen de `evaluation/metrics.py`
(P4) y `experiments/reports.py`; las propias del LLM, de `experiments/llm_analysis.py`.

In [ ]:
from evaluation import metrics
from experiments import reports

LLM_SYSTEMS = [s for s in ("llm_zero", "llm_few") if results_path(s).exists()]
NAMES = {"parser": "Parser (CFG)", "spacy": "spaCy", "transformers": "Transformers",
         "llm_zero": "LLM zero-shot", "llm_few": "LLM few-shot"}
if not LLM_SYSTEMS:
    raise SystemExit("Todavía no hay results/llm_zero.jsonl ni results/llm_few.jsonl: correr con PILOT = None.")
preds = {s: load_predictions(s) for s in ("parser", "spacy", "transformers", *LLM_SYSTEMS) if results_path(s).exists()}
parser_preds = preds.get("parser")
log = read_log()
print("Sistemas:", ", ".join(NAMES[s] for s in preds))

### 3.1 ¿Respondió en el formato pedido?

Se le pidió **solo** el JSON. `json_puro` es la respuesta exacta; `json_con_bloque_de_codigo` lo envuelve en
` ```json `, que se acepta. `sin_json` cuenta como error en todas las métricas (sección 2.6).

In [ ]:
answer_format = pd.DataFrame({NAMES[s]: llm_analysis.answer_format(log, s) for s in LLM_SYSTEMS}).T
answer_format

### 3.2 Pregunta 1: ¿reglas o español?

- `coincide_reglas` / `coincide_espanol`: fracción de decisiones que coinciden con `en_gramatica` y con
  `espanol_valido`. Si la segunda es mayor, el LLM sigue su español y no nuestra gramática.
- `desviacion`: de las 11 oraciones de español correcto que la gramática **no** genera, cuántas acepta.
- `acepta_agramaticales`: de las oraciones que no son español correcto, cuántas acepta.
- `arbol_en_gramatica`: de sus árboles (oraciones aceptadas), cuántos usan solo las formas de
  `TREE_SHAPES`. `arbol_igual_parser`: cuántos son exactamente el árbol de nuestro parser.

Como referencia, los otros dos sistemas que deciden (parser y Transformers) en las mismas dos columnas.

In [ ]:
llm_summary = pd.DataFrame({NAMES[s]: llm_analysis.summary(items, preds[s], parser_preds) for s in LLM_SYSTEMS}).T
llm_summary.to_csv(REPORTS / "llm_resumen.csv")
display(llm_summary.round(3))

reference_rows = []
for s in ("parser", "transformers", *LLM_SYSTEMS):
    if s not in preds:
        continue
    row = {"sistema": NAMES[s]}
    for name, reference in (("en_gramatica", "in_grammar"), ("espanol_valido", "valid_spanish")):
        report = reports.acceptance_report(reports.acceptance_confusion(items, preds[s], reference))
        row.update({f"{metric} ({name})": report[metric]
                    for metric in ("precision", "cobertura", "rechazo_correcto", "exactitud")})
    reference_rows.append(row)
references = pd.DataFrame(reference_rows).set_index("sistema")
references.to_csv(REPORTS / "llm_reglas_vs_espanol.csv")
references.round(3).T

In [ ]:
# Figura para el informe: exactitud de la aceptación contra cada referencia.
import matplotlib.pyplot as plt

from evaluation import plots

fig, (ax,) = plots._figure(max(5.0, 1.6 * len(references) + 2), 3.6)
plots._style(ax)
plots._grouped_bars(
    ax, list(references.index),
    {"aceptacion": list(references["exactitud (en_gramatica)"]),
     "aceptacion_espanol": list(references["exactitud (espanol_valido)"])},
    plots.REFERENCE_COLORS,
    {"aceptacion": "contra en_gramatica (las reglas)", "aceptacion_espanol": "contra espanol_valido (el español)"},
)
plots._legend(fig, ax, 2)
plots._title(fig, "¿Reglas o español? Exactitud de la aceptación según la referencia")
fig.savefig(FIGURES / "llm_reglas_vs_espanol.png", dpi=200, bbox_inches="tight", facecolor=plots.SURFACE)

**Las oraciones que separan reglas de español.** Las 11 de `en_gramatica = no` y `espanol_valido = si`:
si el LLM las acepta, se desvió hacia el español. Se muestra también qué `tipo_error` da cuando rechaza.

In [ ]:
def decision(p) -> str:
    if p is None:
        return "—"
    if p.accepts is None:
        return "sin JSON"
    return "acepta" if p.accepts else f"rechaza ({p.error_type})"


drift = pd.DataFrame([{
    "id": i.id, "oracion": i.sentence, "fenomeno": ", ".join(i.phenomena),
    **{NAMES[s]: decision(next(p for p in preds[s] if p.id == i.id)) for s in ("parser", "transformers", *LLM_SYSTEMS)
       if s in preds},
} for i in items if not i.in_grammar and i.valid_spanish])
drift.to_csv(REPORTS / "llm_desviacion.csv", index=False)
drift

**Sus árboles.** Para las oraciones que acepta: ¿usa solo nuestras formas de árbol? Si no, qué nodo está
mal. Si sí, ¿es el mismo árbol que construye el parser (que es el único que la gramática permite, salvo la
unión de la PP)?

In [ ]:
from spanish.grammar import tree_shape_errors
from spanish.tree import Node, same_tree

parser_tree = {p.id: p.tree for p in parser_preds or [] if p.tree}


def tree_check(p) -> str:
    try:
        errors = tree_shape_errors(Node.from_bracketed(p.tree))
    except ValueError as e:
        return f"no se puede leer: {e}"
    if errors:
        return "; ".join(map(str, errors[:2]))
    if p.id in parser_tree:
        return "igual al parser" if same_tree(p.tree, parser_tree[p.id]) else "en la gramática, distinto al parser"
    return "en la gramática (el parser no tiene árbol)"


tree_rows = [{"sistema": NAMES[s], "id": p.id, "oracion": by_id[p.id].sentence, "arbol": tree_check(p)}
             for s in LLM_SYSTEMS for p in preds[s] if p.accepts and p.tree]
trees = pd.DataFrame(tree_rows)
display(trees.groupby(["sistema", "arbol"]).size().to_frame("oraciones"))
trees[trees["arbol"] != "igual al parser"]

### 3.3 Pregunta 2: ¿dónde acierta y dónde falla?

Exactitud de la decisión contra `en_gramatica` por nivel y por fenómeno, el tipo de error que da al
rechazar, y el resto de las métricas comunes (etiquetas, funciones y unión de la PP).

In [ ]:
def group_table(key: str) -> pd.DataFrame:
    frames = []
    for s in LLM_SYSTEMS:
        rows = llm_analysis.by_group(items, preds[s], key, parser_preds)
        frames.append(pd.DataFrame(rows).set_index("grupo")[["oraciones", "coincide_reglas", "coincide_espanol"]]
                      .rename(columns=lambda c: f"{c} ({s})"))
    return pd.concat(frames, axis=1)


by_level = group_table("level")
by_phenomenon = group_table("phenomena")
by_level.to_csv(REPORTS / "llm_por_nivel.csv")
by_phenomenon.to_csv(REPORTS / "llm_por_fenomeno.csv")
display(by_level.round(3))
by_phenomenon.round(3)

In [ ]:
# Tipo de error: esperado (filas) frente a lo que respondió el LLM (columnas), en las oraciones fuera de la gramática.
for s in LLM_SYSTEMS:
    pairs = metrics.align(items, preds[s])
    counts = metrics.error_type_confusion(pairs)
    table = pd.DataFrame([{"esperado": e, "respuesta": r, "oraciones": n} for (e, r), n in counts.items()])
    print(f"\n=== {NAMES[s]} | tipo_error correcto en los rechazos: {metrics.error_type_accuracy(pairs)}")
    display(table.pivot(index="esperado", columns="respuesta", values="oraciones").fillna(0).astype(int))

In [ ]:
# Métricas comunes, con los otros sistemas al lado (mismas definiciones que results/metricas.md).
metric_rows = metrics.compute_all(metrics.load_all_pairs(items))
val = lambda s, comp, m: metrics.value(metric_rows, s, comp, m)
common = pd.DataFrame({NAMES[s]: {
    "etiquetado: exactitud": val(s, "etiquetado", "exactitud"),
    "etiquetado: oraciones con otro número de palabras": val(s, "etiquetado", "oraciones_distinto_largo"),
    "sujeto": val(s, "funciones", "sujeto"),
    "verbo": val(s, "funciones", "verbo"),
    "objeto": val(s, "funciones", "objeto"),
    "pp_adjunta: exactitud": val(s, "estructura", "pp_adjunta"),
    "pp_adjunta: ambiguas detectadas": val(s, "estructura", "ambiguas_detectadas"),
} for s in preds})
common.to_csv(REPORTS / "llm_metricas_comunes.csv")
common.round(3).astype(object).fillna("—")

In [ ]:
# Unión de la PP: el LLM es el único sistema que puede responder "ambigua".
for s in LLM_SYSTEMS:
    matrix, summary = reports.pp_attachment_report(items, preds[s])
    print(f"\n=== {NAMES[s]}")
    display(matrix)
    display(summary.to_frame("valor").T)

In [ ]:
# Oración por oración: dónde falla cada versión (decisión distinta de en_gramatica).
detail = pd.DataFrame([{
    "id": i.id, "nivel": i.level, "oracion": i.sentence, "fenomeno": ", ".join(i.phenomena),
    "esperado": "acepta" if i.in_grammar else f"rechaza ({i.error_type})",
    **{NAMES[s]: decision(next(p for p in preds[s] if p.id == i.id)) for s in LLM_SYSTEMS},
} for i in items])
detail.to_csv(REPORTS / "llm_por_oracion.csv", index=False)
decision_of = {s: {p.id: p.accepts for p in preds[s]} for s in LLM_SYSTEMS}
wrong = detail[[any(decision_of[s][i.id] != i.in_grammar for s in LLM_SYSTEMS) for i in items]]
print(f"{len(wrong)} oraciones en las que al menos una versión no decide como la gramática")
wrong

### 3.4 Pregunta 3: ¿el few-shot mejora?

Las mismas métricas lado a lado y la diferencia (few − zero), y las oraciones en las que las dos versiones
responden distinto.

In [ ]:
if len(LLM_SYSTEMS) < 2:
    print("Hace falta correr las dos versiones.")
else:
    compare_metrics = {
        "coincide_reglas": lambda s: llm_summary.loc[NAMES[s], "coincide_reglas"],
        "coincide_espanol": lambda s: llm_summary.loc[NAMES[s], "coincide_espanol"],
        "desviacion": lambda s: llm_summary.loc[NAMES[s], "desviacion"],
        "tipo_error_correcto": lambda s: llm_summary.loc[NAMES[s], "tipo_error_correcto"],
        "arbol_en_gramatica": lambda s: llm_summary.loc[NAMES[s], "arbol_en_gramatica"],
        "arbol_igual_parser": lambda s: llm_summary.loc[NAMES[s], "arbol_igual_parser"],
        "precision (en_gramatica)": lambda s: references.loc[NAMES[s], "precision (en_gramatica)"],
        "rechazo_correcto (en_gramatica)": lambda s: references.loc[NAMES[s], "rechazo_correcto (en_gramatica)"],
        **{name: (lambda s, name=name: common.loc[name, NAMES[s]]) for name in common.index},
    }
    few_vs_zero = pd.DataFrame({NAMES[s]: {m: f(s) for m, f in compare_metrics.items()} for s in LLM_SYSTEMS})
    few_vs_zero = few_vs_zero.apply(pd.to_numeric, errors="coerce")
    few_vs_zero["few − zero"] = few_vs_zero[NAMES["llm_few"]] - few_vs_zero[NAMES["llm_zero"]]
    few_vs_zero.to_csv(REPORTS / "llm_few_vs_zero.csv")
    display(few_vs_zero.round(3))

    changed = detail[detail[NAMES["llm_zero"]] != detail[NAMES["llm_few"]]]
    print(f"{len(changed)} oraciones con distinta decisión entre las dos versiones")
    display(changed)

### 3.5 Costo

Tiempo de cada llamada real a la API y tokens que reporta el proveedor (del log; las respuestas reutilizadas
conservan el tiempo de su llamada original). El few-shot manda un prompt más largo: más tokens de entrada
por oración. Para estimar el precio, llenar `PRICE_PER_MILLION` con la tarifa vigente del modelo (dólares por
millón de tokens); en la capa gratuita es 0.

In [ ]:
PRICE_PER_MILLION = {"entrada": None, "salida": None}   # TODO(P5): tarifa vigente de MODEL, o 0 en la capa gratuita

cost = pd.DataFrame(llm_analysis.cost_from_log(log))
if PRICE_PER_MILLION["entrada"] is not None and not cost.empty:
    cost["dolares_por_oracion"] = (cost["tokens_entrada_promedio"] * PRICE_PER_MILLION["entrada"]
                                   + cost["tokens_salida_promedio"] * PRICE_PER_MILLION["salida"]) / 1e6
cost.to_csv(REPORTS / "llm_costo.csv", index=False)
display(cost.round(1))

others = {NAMES[s]: pd.Series([p.time_ms for p in preds[s]]).median()
          for s in ("parser", "spacy", "transformers", *LLM_SYSTEMS) if s in preds}
pd.Series(others, name="ms por oración (mediana)").round(2).to_frame()

### 3.6 Respuestas crudas para citar en el informe

Respuestas del log de algunos casos interesantes: una desviación hacia el español, un rechazo por
concordancia y, si hay, una respuesta sin JSON.

In [ ]:
from experiments.llm_client import find_in_log


def show_raw(system: str, id_: int) -> None:
    entry = find_in_log(id_, system)
    if entry is None:
        return
    print(f"=== {NAMES[system]} | id {id_}: {by_id[id_].sentence} | modelo {entry['modelo']} | {entry['fecha']}")
    print(entry["respuesta_cruda"].strip()[:1500])
    print()


for s in LLM_SYSTEMS:
    by_pred = {p.id: p for p in preds[s]}
    drifted = [i.id for i in items if not i.in_grammar and i.valid_spanish and by_pred[i.id].accepts]
    agreement = [i.id for i in items if i.error_type == "concordancia"]
    no_json = [p.id for p in preds[s] if p.run_error == "respuesta_no_json"]
    for id_ in drifted[:1] + agreement[:1] + no_json[:1]:
        show_raw(s, id_)

### 3.7 Conclusiones (para el informe)

TODO(P5), con las cifras de las secciones 3.1 a 3.5:

1. ¿Respeta las reglas o se desvía? Comparar `coincide_reglas` con `coincide_espanol` y citar ejemplos del log
   (sección 3.6).
2. ¿Dónde acierta y dónde falla? Por nivel y por fenómeno (sección 3.3).
3. ¿El few-shot mejora la precisión? Columna `few − zero` (sección 3.4).
4. Para la comparación de enfoques: robustez, ambigüedad (`pp_adjunta = ambigua`), escalabilidad y costo.